In [0]:
catalog_name = "ct_oil_gas" 
bronze_table_name = "oil_gas_transactions"
bronze_schema_name = "sc_bronze"
silver_schema_name = "sc_silver"
silver_table_name = "oil_gas_transactions"


In [0]:
df = spark.table(f"{catalog_name}.{bronze_schema_name}.{bronze_table_name}")

In [0]:
%run ./nb_transformation

In [0]:
%run ./nb_dq_checks

In [0]:
import logging
from pyspark.sql.functions import current_timestamp


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

try:
    logger.info("Loading transformation config...")
    config_rows = load_transformation_config(spark, 1)

    logger.info(f"Reading bronze table: {catalog_name}.{bronze_schema_name}.{bronze_table_name}")
    df = spark.table(f"{catalog_name}.{bronze_schema_name}.{bronze_table_name}")

    logger.info("Applying transformations...")
    transformer = Transformation(df)
    transformed_df = transformer.apply_config(config_rows)

except Exception as e:
    logger.error(f"Transformation stage failed: {e}")
    raise

try:
    logger.info("Loading DQ rules config...")
    config_rows = load_dq_rules(spark, job_id=1)

    logger.info("Running DQ checks...")
    checker = DQChecker(transformed_df)
    valid_df, invalid_df = checker.apply_config(config_rows)  

except Exception as e:
    logger.error(f"DQ check stage failed: {e}")
    raise

try:
    logger.info(f"Writing invalid rows to quarantine: {catalog_name}.{silver_schema_name}.{silver_table_name}_dq_quarantine")
    invalid_df.write.format("delta").mode("append").saveAsTable(
        f"{catalog_name}.{silver_schema_name}.{silver_table_name}_dq_quarantine"
    )
except Exception as e:
    logger.error(f"Failed to write quarantine table: {e}")
    raise

try:
    logger.info(f"Writing valid rows to silver: {catalog_name}.{silver_schema_name}.{silver_table_name}")
    valid_df.withColumn("silver_timestamp", current_timestamp()) \
        .write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{catalog_name}.{silver_schema_name}.{silver_table_name}")
except Exception as e:
    logger.error(f"Failed to write silver table: {e}")
    raise

logger.info("Silver notebook completed successfully.")